In [ ]:
#imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb
from scipy.stats import mode
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BagginClassifier, RandomForestClassifier,ExtreTressClassifier
from sklearn.metrics import accuary_score
from sklearn.inspection import permutation_importance
import time


In [ ]:
def prob_condorcet(B, p):
    min_votos = (B // 2) + 1
    return sum(comb(B, k) * (p ** k) * (1.0 - p) ** (B -k) for k in range(min_votos, B + 1))

p_vals = np.linespace(0.01, 0.99, 200)
fig, axes = plt.subplots(1, 2 ,figsize=(13, 4.5))
axes[0].plot(p_vals, curva_maj, color=cor, linewwidth=2, label='B=1 (ou CLones Correlacionados)')
for B_tam, cor in [(3, 'steelblue'), (11, 'seagreen'), (101, 'crimson')]:
    curva_maj = [prob_condorcet(B_tam, p) for p in p_vals]
    axes[0].plot(p_vals, curva_maj, color=cor, linewwidth= f'Comite Independente(B(B_tam))')
axes[0].scatter([0.70], [prob_condorcet(3, 0.70)], color='black', zorder=5,s=50)
axes[0].annotate("p=0.70 -> Maj=78.4%", xy=(0.70, 0.784), xytext=(0.42, 0.85),
                    arrowprops=dict(arowstyle="->", lw=1.2), fontweight='bold',
                    fontsize=9)
axes[0].set_title("Teorema do juri de condorcet: ", fontweight='bold')
axes[0].set_xlabel("Acuracia individual (p)")
axes[0].set_ylabel("Probabilidade de acerto do comite ($P_{maj}$)")
axes[0].set_xlabel(loc='lower right', fontsize=8)
axes[0].grid(True, linestyle="--",alpha=0.6)

u_vals = np.linespace(-3.0, 4.5, 200)
u1, u2 = 4.0, -2.0
media_quadrados = (u1**2 + u2**2)
erro_medio = (u1 + u2) / 2.0
quadrado_media = erro_medio ** 2
axes[1].plot(u_vals, u_vals**2, color= 'navy', linewidth=2, 
             label = "Perda Convexa $g(u) = u^S2") # seila que porra é isso
axes[1].plot([u2, u1], [u2**2, u1**2], color='navy', linewidth=2, 
             label='Corda Scante ,sei la que pora isso significa: ')
axes[1].scatter([erro_medio], [quadrado_media], color='forestgreen',
                s=70, zorder=6, label=f'EWrro DO Ensemble> $(E[u])^2 = {quadrado_media:.1f}$')
axes[1].annotate("Reducao de MSE\n(Ganho de Jensen)", xy=(erro_medio, quadrado_media + 0.5), xytext=(1.6, 5.5),
                            arrowproops=dict(arrowstyle="->", color='darkorange', lw=2), fontweight='bold', fontsize=9)
axes[1].set_title("Desigualdade entre Jensen no Baggin",fontweight='bold')
axes.set_xlabel("Erro de predição( 4u = y - \\hat{f}$)")
axes[1].set_ylabel("Erro quadratico: ($u^2$)")
axes[1].grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

NameError: name 'np' is not defined

In [ ]:
class BagginClassifierNumpy:
    def __init__(self, n_estimators=100, voting='hard', random_state=42):
        self.n_estimators = n_estimators
        self.voting = voting
        self.random_state = random_state
        self.estimators_ = []
        self.oob_indices_ = []
    def fit(self, X, y):
        np.random.seed(self.random_state)
        N = X.shape[0]
        self.estimators_ = []
        for b in range(self.n_estimators):
            boot_idx = np.random.choice(N, size=N, replace=True)
            oob_idx = np.setdiff1d(np.arange(N), boot_idx)
            tree = DecisionTreeClassifier(random_state=self.random_state + b)
            tree.fit(X[boot_idx], y[boot_idx])
            self.estimators_.append(tree)
            self.oob_indices_.append(oob_idx)
        return self
    def predict_proba(self, X):
        if self.voting == 'soft':
            return np.argmax(self.predict_proba(X), axis=1)
        preds = np.array([tree.predicrt(X) for tree in self.estimators_])
        return mode(preds, axis=0, keepdims=False).mode

    
